# 웹 페이지 로더

웹 페이지의 HTML 콘텐츠를 크롤링하여 Document 객체로 변환하는 방법을 학습합니다.

`WebBaseLoader`는 웹 기반 문서를 로드하는 범용 로더로, BeautifulSoup을 사용하여 HTML을 파싱하고 텍스트를 추출합니다.

온라인 뉴스, 블로그, 기술 문서 등을 RAG 시스템에 활용할 수 있습니다.

# 07. 웹 페이지 로더

## 학습 목표
- `WebBaseLoader`로 URL에서 HTML 콘텐츠를 가져와 Document로 변환해요
- `SoupStrainer`로 필요한 HTML 요소만 선택적으로 파싱해요
- 여러 URL을 동시에 비동기로 로딩하는 방법을 익혀요

## 사전 지식
- 00-Document-Loader 노트북 완료
- `pip install beautifulsoup4 requests` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — 파일이 아닌 **웹 URL**에서 직접 데이터를 가져와요.

In [15]:
from dotenv import load_dotenv

# 예제 URL (위키백과 - 기계 학습 문서)

# 아래에 코드를 작성하세요
URL = "https://popcateum.org"
TEST_URL = "https://en.wikipedia.org/wiki/Machine_learning"


## 1. WebBaseLoader 기본 사용

`WebBaseLoader`는 웹 페이지의 HTML을 다운로드하고 BeautifulSoup으로 파싱하여 텍스트를 추출합니다.

**필요한 패키지**: `pip install beautifulsoup4 requests`

**주요 특징**:
- HTML 자동 파싱
- 메타데이터 자동 추출 (title, language 등)
- 단일/복수 URL 지원

In [19]:
# ============================================================
# TODO: WebBaseLoader로 웹 페이지를 Document로 로드해보세요
# 힌트: WebBaseLoader(TEST_URL) → loader.load()
# 예상 결과: 1개의 Document가 생성되고 title 등 메타데이터가 포함됩니다
# ============================================================

from langchain_community.document_loaders import WebBaseLoader

# 1단계: WebBaseLoader 생성
# - URL을 전달하면 HTML을 자동으로 다운로드하고 BeautifulSoup으로 파싱
loader = WebBaseLoader(TEST_URL)
# 2단계: 웹 페이지 로드
docs = loader.load()
# 3단계: 결과 확인
docs[0].page_content[:200]


'\n\n\n\nMachine learning - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout '

## 2. BeautifulSoup으로 특정 요소만 추출

`bs4.SoupStrainer`를 사용하면 HTML의 특정 요소만 선택적으로 파싱할 수 있습니다.

이를 통해 불필요한 네비게이션, 광고, 푸터 등을 제외하고 핵심 콘텐츠만 추출할 수 있습니다.

**시나리오**: 기술 블로그에서 본문(`article` 태그)만 추출하기

In [25]:
# ============================================================
# TODO: SoupStrainer로 Wikipedia 본문 영역만 추출해보세요
# 힌트: bs_kwargs=dict(parse_only=bs4.SoupStrainer("div", attrs={"class": ["mw-parser-output"]}))
# 예상 결과: 전체 페이지보다 짧은 텍스트 길이가 출력됩니다
# ============================================================

import bs4

# 기술 블로그 예제 (실제 URL로 변경 가능)
URL = "https://popcateum.org"
TEST_URL = "https://en.wikipedia.org/wiki/Artificial_intelligence"
# 1단계: SoupStrainer로 파싱할 요소 지정
# - bs_kwargs: BeautifulSoup에 전달할 추가 옵션
# - parse_only: 이 필터에 해당하는 요소만 파싱 (나머지 HTML은 무시)
loader = WebBaseLoader(
    web_path=TEST_URL,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["mw-content-ltr mw-parser-output"]}
        )
    )
)

docs = loader.load()
docs[0].page_content

# 2단계: 로드 및 확인

print(docs[0].page_content)


Intelligence of machines
"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).


Part of a series onArtificial intelligence (AI)
Major goals
Artificial general intelligence
Intelligent agent
Recursive self-improvement
Planning
Computer vision
General game playing
Knowledge representation
Natural language processing
Robotics
AI safety

Approaches
Machine learning
Symbolic
Deep learning
Bayesian networks
Evolutionary algorithms
Hybrid intelligent systems
Systems integration
Open-source
AI data centers

Applications
Bioinformatics
Deepfake
Earth sciences
 Finance 
Generative AI
Art
Audio
Music
Government
Healthcare
Mental health
Industry
Software development
Translation
 Military 
Physics
Projects

Philosophy
AI alignment
Artificial consciousness
The bitter lesson
Chinese room
Friendly AI
Ethics
Existential risk
Turing test
Uncanny valley
Human–AI interaction

History
Timeline
Progress
AI winter
AI boom
AI bubble

Controversies
Deepfake

## 3. User-Agent 설정 및 SSL 우회

일부 웹사이트는 크롤러를 차단하거나 SSL 인증을 요구할 수 있습니다.

이 경우 `header_template`으로 User-Agent를 설정하거나, `requests_kwargs`로 SSL 인증을 우회할 수 있습니다.

In [26]:
# User-Agent를 설정하여 일반 브라우저처럼 위장
loader = WebBaseLoader(
    TEST_URL,
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36",
    }
)
# SSL 인증 우회 (필요한 경우)
loader.requests_kwargs = {"verify": True}

# 아래에 코드를 작성하세요
docs = loader.load()

## 4. 여러 URL 동시에 로드

여러 웹 페이지를 한 번에 로드할 수 있습니다.

URL 리스트를 전달하면, 각 URL이 개별 Document로 변환됩니다.

> **실무 팁**: JavaScript로 렌더링되는 SPA(React/Vue 기반) 페이지는 `WebBaseLoader`로 가져오면 빈 내용이 나옵니다. 그런 경우에는 Playwright나 Selenium 기반 로더를 사용해야 합니다.

In [ ]:
# ============================================================
# TODO: URL 리스트를 WebBaseLoader에 전달해서 여러 페이지를 한 번에 로드해보세요
# 힌트: WebBaseLoader(urls) → loader.load() → 각 URL이 별도 Document로 변환됨
# 예상 결과: 3개의 Document가 생성되고 각각의 source URL이 메타데이터에 포함됩니다
# ============================================================

urls = [
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Computer_vision"
]

# 1단계: 여러 URL을 리스트로 전달
loader = WebBaseLoader(
    urls
)

docs = loader.load()

d1_title = docs[0].metadata.get("title")
print(f' ==> [Line 20]: \033[38;2;5;226;207m[d1_title]\033[0m({type(d1_title).__name__}) = \033[38;2;241;171;173m{d1_title}\033[0m')
d2_title = docs[1].metadata.get("title")
print(f' ==> [Line 22]: \033[38;2;13;161;2m[d2_title]\033[0m({type(d2_title).__name__}) = \033[38;2;16;153;100m{d2_title}\033[0m')
d3_title = docs[2].metadata.get("title")
print(f' ==> [Line 24]: \033[38;2;2;146;160m[d3_title]\033[0m({type(d3_title).__name__}) = \033[38;2;21;4;100m{d3_title}\033[0m')





 ==> [Line 20]: [d1_title](str) = Deep learning - Wikipedia
 ==> [Line 22]: [d2_title](str) = Natural language processing - Wikipedia
 ==> [Line 24]: [d3_title](str) = Computer vision - Wikipedia


## 5. 비동기 로딩 (성능 향상)

여러 URL을 동시에 스크래핑하면 처리 속도를 크게 향상시킬 수 있습니다.

`aload()` 메서드를 사용하면 비동기로 여러 페이지를 병렬 다운로드합니다.

**주의**: `requests_per_second`로 초당 요청 수를 제한하여 서버 부하를 조절해야 합니다.

In [ ]:
import asyncio
import nest_asyncio
from typing import List

from langchain_core.documents import Document

# Jupyter Notebook에서 asyncio 사용을 위해 필요

# 여러 URL 준비


# LangChain 0.3.14+에서는 aload가 동기 메서드로 변경되어 alazy_load를 직접 사용해야 함

# 비동기 로드

# 아래에 코드를 작성하세요


## 6. 프록시 설정

IP 차단을 우회하거나 특정 네트워크 환경에서 크롤링할 때 프록시를 사용할 수 있습니다.

`proxies` 파라미터로 HTTP/HTTPS 프록시 서버를 지정합니다.

## 핵심 정리 및 마무리

### WebBaseLoader 주요 파라미터

| 파라미터 | 설명 | 예시 |
|---------|------|------|
| `web_paths` | URL(s) | `"https://..."` 또는 리스트 |
| `bs_kwargs` | BeautifulSoup 옵션 | `dict(parse_only=SoupStrainer(...))` |
| `header_template` | HTTP 헤더 | `{"User-Agent": "..."}` |
| `requests_per_second` | 초당 요청 수 제한 | `2` |

### 실무 팁
> 웹사이트를 크롤링할 때는 반드시 `robots.txt`를 확인하고 이용 약관을 준수해야 해요. `requests_per_second`를 2 이하로 설정해서 서버에 부담을 주지 않는 것이 중요해요.

### 주의 사항
> JavaScript로 동적으로 렌더링되는 페이지(SPA 등)는 `WebBaseLoader`로 가져올 수 없어요. 그런 경우에는 Selenium이나 Playwright가 필요해요.

---

## 다음 예고

다음은 가장 단순하지만 한국어 처리에 주의가 필요한 텍스트 파일 로더를 배워볼게요.

- **08-TXT-Loader**: 인코딩 처리 핵심 (utf-8, cp949, euc-kr)
- **09-JSON-Loader**: 구조화된 데이터에서 특정 필드 추출